In [1]:
import os 
import numpy as np 

In [2]:

train_data = np.memmap("data/train.bin", dtype=np.uint16, mode="r")
val_data   = np.memmap("data/val.bin",   dtype=np.uint16, mode="r")

In [3]:
import torch
block_size = 8
batch_size = 32
vocab_size = 4096
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i : i + block_size].astype(np.int64))         for i in ix.tolist()])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + block_size].astype(np.int64)) for i in ix.tolist()])
    return x.to(device), y.to(device)

In [9]:
import torch.nn.functional as F
from torch import nn 

class BigramModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
    self.positional_embedding_table = nn.Embedding(block_size, n_embd)
    self.lm_head = nn.Linear(n_embd, vocab_size)
  def forward(self, idx, targets=None):
          B, T = idx.shape  # (B, T)
          tok_emb = self.token_embedding_table(idx) # (B,T, N_EMBD)
          pos_emb = self.positional_embedding_table(torch.arange(T, device = device)) # T,N_EMBD
          x = tok_emb + pos_emb
          logits = self.lm_head(x) # (B,T,vocab_size)

          loss = None
          if targets is not None:
              B, T, V = logits.shape
              loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
          return logits, loss
  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
          # idx: (B, T) tensor of token ids to start from
          for _ in range(max_new_tokens):
              idx_cond = idx[:, -block_size:]
              logits, _ = self(idx_cond)
              logits = logits[:, -1, :]
              probs = F.softmax(logits, dim=-1)
              next_id = torch.multinomial(probs, 1)
              idx = torch.cat([idx, next_id], dim=1)
          return idx

In [5]:
model = BigramModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [6]:
@torch.no_grad()
def estimate_loss():
        out = {}
        model.eval()
        for split in ["train", "val"]:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                x, y = get_batch(split)
                _, loss = model(x, y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

In [7]:
for step in range(max_iters):

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train {losses['train']:.4f}, val {losses['val']:.4f}")

    x, y = get_batch("train")                # 1. get data
    logits, loss = model(x, y)               # 2. forward pass
    optimizer.zero_grad(set_to_none=True)    # 3. clear old gradients
    loss.backward()                          # 4. compute new gradients
    optimizer.step()                         # 5. update the table

step 0: train 8.6720, val 8.6869
step 300: train 4.6573, val 4.6530
step 600: train 4.3690, val 4.3627
step 900: train 4.2153, val 4.2193
step 1200: train 4.1426, val 4.1370
step 1500: train 4.0613, val 4.0964
step 1800: train 4.0661, val 4.0647
step 2100: train 4.0444, val 4.0217
step 2400: train 4.0123, val 4.0127
step 2700: train 4.0203, val 3.9773
step 2999: train 3.9661, val 3.9917


In [12]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("tokenizer")
EOT_ID = tok.convert_tokens_to_ids("<|endoftext|>")   # 0

start = torch.tensor([[EOT_ID]], dtype=torch.long, device=device)   # shape (1, 1)
out = model.generate(start, max_new_tokens=100)                     # shape (1, 101)
print(tok.decode(out[0].tolist()))

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
